# ESPIn 2026 Tidal Sed

Jana Frenzel, Marloes Bonenkamp, Hannah Henry, Maya Maes-Johnson, Anderson Amaya Saldarriaga

## Import Landlab + Components + others

In [ ]:
# landlab imports
from landlab import RasterModelGrid, imshow_grid
from landlab.components import TidalFlowCalculator
from landlab.plot.graph import plot_graph
from landlab.grid.mappers import map_link_vector_components_to_node, map_mean_of_link_nodes_to_link

# bmi imports
from bmi_topography import Topography

# other imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.axes_grid1 import make_axes_locatable
from IPython.display import Image

## Import Real Elevation Data and Create a Grid

Documentation for bmi-topography can be found at https://bmi-topography.csdms.io/en/latest/#api-key

Values for your south, north, west, and east values can be calculated at https://mapscaping.com/bounding-box-calculator/ 

To access OpenTopography's elevation data, you will need to create an API key. Information for how to do this can be found at: https://opentopography.org/blog/introducing-api-keys-access-opentopography-global-datasets

In [ ]:
# Import elevation data from OpenTopography -- this example is of Hog Island, VA
barrier_island_dem = Topography(
    dem_type = "SRTM15Plus", 
    south = 37.362517, 
    north = 37.473768, 
    west = -75.761307, 
    east = -75.623978, 
    api_key = 'YOUR API KEY HERE')

barrier_island = barrier_island_dem.load()
barrier_island.values.shape

"""
Example Topographies:

Gulf of Uraba, Colombia
-----------------------
    south = 7.895485, 
    north = 8.179684,
    west = -76.954559,
    east = -76.696381,

Hog Island, VA
-----------------------
    south = 37.362517, 
    north = 37.473768, 
    west = -75.761307, 
    east = -75.623978, 
"""

In [ ]:
# Plot elevation as a check
import matplotlib.pyplot as plt
barrier_island_dem.da.plot(cmap="terrain")
plt.show()

RasterModelGrid() takes the parameter, xy_spacing, this is the resolution of your elevation data (SRTMGL3: 90 x 90 m, SRTM15Plus: 500 x 500 m, etc.)

In [ ]:
# Input elevation data into grid
z = np.flipud(barrier_island.values.squeeze())
z = z.astype(float)
grid = RasterModelGrid(z.shape, xy_spacing=(500.0, 500.0))
grid.at_node["topographic__elevation"] = z

In [ ]:
# set the tidal boundarys, where you want the water to come from
grid.set_closed_boundaries_at_grid_edges(False, True, True, True)

"""
Example Topographies:

grid.set_closed_boundaries_at_grid_edges(East, North, West, South)

Gulf of Uraba, Colombia
-----------------------
grid.set_closed_boundaries_at_grid_edges(True, False, True, True)

Hog Island, VA
-----------------------
grid.set_closed_boundaries_at_grid_edges(False, True, True, True)

"""

In [ ]:
# Show grid
grid.imshow("topographic__elevation", cmap="terrain", vmin=-6) # vmin may need to be less than 0 if bathymetric data is loaded

## Calculate cycle-averaged tidal flow
The next section implements the landlab Tidal Flow component: https://landlab.readthedocs.io/en/latest/generated/api/landlab.components.tidal_flow.tidal_flow_calculator.html 

In [ ]:
# tidal parameters
tidal_range = 5
#tidal_
roughness = 0.01

# create instance
tfc = TidalFlowCalculator(grid, tidal_range=tidal_range, roughness=roughness)

In [ ]:
# run model
tfc.run_one_step() 
# output:
# nodes: topographic elevation and mean water depth
# links: tidal flow velocity (ebb/flood)

# --> we need to use an incision component: e.g. fluvial incision?

In [ ]:
# grid variables
grid.at_link.keys()
grid.at_node.keys()

In [ ]:
# check values at links of field flood tide velocity
grid.at_link["flood_tide_flow__velocity"][30]

In [ ]:
# check values at links of field flood tide velocity
grid.at_link["ebb_tide_flow__velocity"]

## Choose Ebb or Flood Tide

In [ ]:
tide_direction = input('Enter "F" for Flood or "E" for Ebb: ')
if tide_direction == 'F':
    print('Flood Tide')
    title_ID = 'Flood'
    vel_link = grid.at_link["ebb_tide_flow__velocity"]
elif tide_direction == 'E':
    print('Ebb Tide')
    title_ID = 'Ebb'
    vel_link = grid.at_link["flood_tide_flow__velocity"]
else:
    print('Invalid input. Please enter "F" or "E"')

In [ ]:
# map links and nodes
vel_x, vel_y = map_link_vector_components_to_node(grid, vel_link)

In [ ]:
# Get node coordinates
node_x = grid.node_x
node_y = grid.node_y

In [ ]:
# test plot
# Plot using matplotlib quiver
grid.imshow("topographic__elevation", cmap="terrain", vmin=-6) # vmin may need to be less than 0 if bathymetric data is loaded
plt.quiver(node_x, node_y, vel_x, vel_y)
plt.xlabel("X Coordinate (m)")
plt.ylabel("Y Coordinate (m)")
plt.title(f"Flow Velocity Mapped from Links to Nodes: {title_ID} Direction ", fontsize = 10)
plt.show()

## Animation

In [ ]:
# Move velocities from links to nodes
vel_x, vel_y = map_link_vector_components_to_node(grid, grid.at_link["ebb_tide_flow__velocity"])

# Get the 2D shape of the grid
shape = grid.shape
X, Y = grid.node_x.reshape(shape), grid.node_y.reshape(shape)
U, V = vel_x.reshape(shape), vel_y.reshape(shape)

# Get the elevation data for the background
Z = grid.at_node["topographic__elevation"].reshape(shape)

# 2. SETUP THE FIGURE
fig, ax = plt.subplots(figsize=(8, 6))
ax.set_aspect('equal') # Keep proportions real

# Draw the background map (this stays still)
img = ax.imshow(Z, origin='lower', extent=[X.min(), X.max(), Y.min(), Y.max()],
                cmap='terrain', vmin=-6)

# Draw the arrows (Tide)
# step=2 means we draw 1 arrow every 2 cells.
step = 2
Q = ax.quiver(X[::step, ::step], Y[::step, ::step], U[::step, ::step], V[::step, ::step],
              color='black', scale=15.0, width=0.003)

# Add the colorbar for the elevation
cax = make_axes_locatable(ax).append_axes("right", size="5%", pad=0.1)
fig.colorbar(img, cax=cax, label="Elevation (m)")
title = ax.set_title("Tidal Flow - Hour 0.0")

# 3. ANIMATION LOOP (30 frames = 12 hours)
def update(frame):
    # Calculate time and the wave (sine)
    time = (frame / 30) * 12.0
    phase = np.sin(2 * np.pi * time / 12.0)

    # Change ONLY the arrows (the map stays the same)
    Q.set_UVC(U[::step, ::step] * phase, V[::step, ::step] * phase)
    title.set_text(f"Tidal Flow - Hour {time:.1f} / 12.0")

    return Q, title

# 4. SAVE THE VIDEO
ani = animation.FuncAnimation(fig, update, frames=30, blit=True)
ani.save("barrier_island_tide.gif", writer=animation.PillowWriter(fps=5))
plt.show()
print("Done!")

## Ebb-Tide

In [ ]:
# save the velocity information
ebb_vel_link = grid.at_link["ebb_tide_flow__velocity"]

In [ ]:
# map links and nodes
ebb_vel_x, ebb_vel_y = map_link_vector_components_to_node(grid, ebb_vel_link)

In [ ]:
# Get node coordinates
ebb_node_x = grid.node_x
ebb_node_y = grid.node_y

In [ ]:
# test plot
# Plot using matplotlib quiver
grid.imshow("topographic__elevation", cmap="terrain", vmin=-6) # vmin may need to be less than 0 if bathymetric data is loaded
plt.quiver(ebb_node_x, ebb_node_y, ebb_vel_x, ebb_vel_y)
plt.xlabel("X Coordinate (m)")
plt.ylabel("Y Coordinate (m)")
plt.title("Flow Velocity Mapped from Links to Nodes")
plt.show()

## Flood-Tide

In [ ]:
# save the flood velocity information
from landlab.grid.mappers import map_link_vector_components_to_node
flood_vel_link = grid.at_link["flood_tide_flow__velocity"]

In [ ]:
# map links and nodes
flood_vel_x, flood_vel_y = map_link_vector_components_to_node(grid, flood_vel_link)

In [ ]:
# get node coordinates
flood_node_x = grid.node_x
flood_node_y = grid.node_y

In [ ]:
# test plot using quiver
grid.imshow("topographic__elevation", cmap="terrain", vmin=-6)
plt.quiver(flood_node_x, flood_node_y, flood_vel_x, flood_vel_y)
plt.xlabel("X Coordinate (m)")
plt.ylabel("Y Coordinate (m)")
plt.title("Flow Velocity Mapped from Links to Nodes")
plt.show()

## Writing a component for sediment transport in a tidal flat

Component consits of three helper functions `calc_deposition()`, `calc_shear()` and `calc_erosion()`, which are called by the main function `calc_bed_evolution()`. 

In development phase, this component only requires a grid object with tidal velocity field produced by the `TidalFlowCalculator` component. Optionally, parameters such as, e.g. sea level rise, diffusion coefficient, can be changed.

In [ ]:
def calc_deposition(w_s: float=0.03, c: float=0.1) -> float:
    """ compute deposition 
    
    PARAMETERS
    w_s: settling velocity, default: fine sand 0.03 [m/s]
    c: sediment concentration, default: SSC in tidal water 0.1 [kg/m3]
    """
    return w_s*c

def calc_shear(grid,v):
    """ computes shear stress at links
    
    PARAMETERS
    grid: grid
    v: velocity vectors (on links) - output from TidalFlowCalculator component

    CONSTANTS
    n: Manning's n, default: 0.035 (for open water)
    rho_w: water density, default: 1025[kg/m3]
    g: gravitational constant 9.81[m/s2]
    """

    n = 0.035 # mannings n for open water
    rho_w = 1025
    g = 9.81

    # water depth is transferred from nodes to links, v is on links
    shear=(
        rho_w*
        g*
        (n**2)*
        (map_mean_of_link_nodes_to_link(grid, "mean_water__depth")**(-1/3))*
        np.sign(v)*(v**2)
    )
    return shear

def calc_erosion(grid,v):
    """ compute erosion 

    DEPENDENCIES
    calc_shear()
    
    PARAMETERS
    grid: grid
    v: velocity vectors (on links) - output from TidalFlowCalculator component

    CONSTANT
    shear_crit: critical shear stress for fine sand, default: 0.2[Pa]
    m_e: erodibility coefficient for fine sand, default: 10^-3 
    
    """
    
    shear_crit = 0.2
    m_e = (10**-3) 

    erosion=m_e * np.sqrt((1+(calc_shear(grid,v)/shear_crit))**2)-1

    return erosion


def calc_bed_evolution(grid, mu: float=3.65, rslr=0):
    """
    calculates bed evolution

    PARAMETERS
    grid
    mu: diffusion coefficient, default: 3.65 (Kirwan and Murray, 2007)
    rslr: relative sea level rise, default: 0 == no sea level change

    DEPENDENCIES
    calc_deposition()
    calc_shear()
    calc_erosion()
    
    CONSTANTS
    rho_s: soil density, default: 2650[kg/m3]

    
    """
    rho_s=2650

    # deposition
    d = calc_deposition()
    # erosion
    e_plus = calc_erosion(grid, v=grid.at_link["flood_tide_flow__velocity"])
    e_min = calc_erosion(grid, v=grid.at_link["ebb_tide_flow__velocity"])
    e = (e_plus + e_min)/2
    e_at_nodes = grid.map_sum_of_inlinks_to_node(e) + grid.map_sum_of_outlinks_to_node(e)
    # calculate elevation change (erosion)
    dz_dt = (
        grid.calc_flux_div_at_node(
            mu*grid.calc_grad_at_link(grid.at_node["topographic__elevation"])
        ) 
        + rslr 
        + ((d-e_at_nodes)/rho_s)
    )
    
    return dz_dt

## Run the bed evolution [insert really cool acronym here] model for one timestep

In [ ]:
dz_dt = calc_bed_evolution(grid)

## Plot results after one model run

In [ ]:
plt.title("Elevation after one tidal cycle")
z_new = (grid.at_node["topographic__elevation"]+dz_dt)
imshow_grid(grid,z_new,cmap="terrain", vmin=-6)

In [ ]:
plt.title("Change in elevation")
imshow_grid(grid,dz_dt,cmap="Greens")

## Coupled Tidal Flow - Bed Evolution model 

### Set up model run parameters

In [ ]:
# set up time-stepping: one timestep is one tidal cycle
number_of_tidal_cycles = 4000
plot_interval = 1000

### Loop over coupled model

In [ ]:
for i in range(0,number_of_tidal_cycles + 1):
    # Compute tidal flow velocity and mean water depth based on TidalFlowCalculator
    tfc.run_one_step()

    # Compute bed level change based on tidal flow characteristics
    dz_dt = calc_bed_evolution(grid)

    # Update bathymetry
    grid.at_node["topographic__elevation"] += dz_dt

    if i % plot_interval == 0:
        print(f"tidal cycle number: {i}")

        # save the flood velocity information
        flood_vel_link = grid.at_link["flood_tide_flow__velocity"]
        # map links and nodes
        flood_vel_x, flood_vel_y = map_link_vector_components_to_node(grid, flood_vel_link)
        # get node coordinates
        flood_node_x = grid.node_x
        flood_node_y = grid.node_y
        
        #Plot bathymetry
        plt.figure()
        imshow_grid(grid, grid.at_node["topographic__elevation"] ,cmap="terrain", vmin=-6)
        plt.show()

        #Plot bed level change
        plt.figure()
        imshow_grid(grid, dz_dt ,cmap="Greens", vmin = 0.001, vmax = 0.0016)
        plt.quiver(flood_node_x, flood_node_y, flood_vel_x, flood_vel_y)
        plt.xlabel("X Coordinate (m)")
        plt.ylabel("Y Coordinate (m)")
        plt.title("Flow Velocity Mapped from Links to Nodes")
        plt.show()